# Rewrite Mode

Instead of replacing entities with tokens, rewrite mode generates a
privacy-safe paraphrase of the entire text. The pipeline:

1. Detects entities (same as replace mode)
2. Classifies the domain and assigns sensitivity dispositions
3. Generates a rewritten version that obscures sensitive entities
4. Evaluates quality (utility) and privacy (leakage) with an automated repair loop
5. Runs a final LLM judge for informational scores

The result includes **utility_score**, **leakage_mass**, and a
**needs_human_review** flag so you can triage records that need attention.

## Setup

In [1]:
from pathlib import Path

try:
    NOTEBOOK_SOURCE_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_SOURCE_DIR = Path.cwd().parent / "notebook_source"

from anonymizer import Anonymizer, AnonymizerConfig, AnonymizerInput, LoggingConfig, Rewrite, configure_logging

configure_logging(LoggingConfig.debug())

In [2]:
anonymizer = Anonymizer()

[12:08:14] [INFO] 🔧 Anonymizer initialized with 3 model configs
[12:08:14] [INFO]   |-- 🔎 detector:  gliner-pii-detector
[12:08:14] [INFO]   |-- ✅ validator: gpt-oss-120b
[12:08:14] [INFO]   |-- 🧩 augmenter: gpt-oss-120b
[12:08:14] [DEBUG] NDD adapter: artifact_path=.anonymizer-artifacts


## Input data

In [3]:
input_data = AnonymizerInput(
    source=str(NOTEBOOK_SOURCE_DIR / "data" / "synth_bios_sample10.csv"),
    text_column="bio",
    data_summary="Biographical profiles",
)

## Basic rewrite

`Rewrite()` with no arguments uses sensible defaults:
conservative risk tolerance, up to 2 repair iterations, and an
auto-populated privacy goal.

In [4]:
config = AnonymizerConfig(rewrite=Rewrite())

preview = anonymizer.preview(
    config=config,
    data=input_data,
    num_records=10,
)

preview.display_record(0)

[12:08:25] [INFO] 📂 Loaded 10 records from /Users/lramaswamy/Documents/github/Anonymizer/docs/notebook_source/data/synth_bios_sample10.csv (column: 'bio')
[12:08:25] [DEBUG] input text lengths: min=1529, max=1843, mean=1661 chars (10 records)
[12:08:25] [DEBUG] detection config: threshold=0.30, labels=(default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)
[12:08:25] [INFO]   |-- 👀 Preview mode: processing 10 of 10 records
[12:08:25] [INFO] 🔍 Running entity detection on 10 records
[12:08:25] [DEBUG] detection aliases: detector=gliner-pii-detector, validator=gpt-oss-120b, augmenter=gpt-oss-120b


[12:08:25] [DEBUG] NDD workflow 'entity-detection' starting with 10 records
[12:08:25] [DEBUG] NDD workflow 'entity-detection': 10 columns ['_raw_detected_entities', '_seed_entities', '_validation_candidates', '_validation_skeleton', '_validation_decisions', '_validated_entities', '_seed_entities_json', '_augmented_entities', '_merged_entities', '_detected_entities']
[12:08:25] [INFO] 🖼️ Preview generation in progress
[12:08:32] [INFO] ✅ Validation passed
[12:08:32] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[12:08:32] [INFO] 🩺 Running health checks for models...
[12:08:32] [INFO]   |-- 👀 Checking 'openai/gpt-oss-120b' in provider named 'nvidia' for model alias 'gpt-oss-120b'...
[12:08:34] [INFO]   |-- ✅ Passed!
[12:08:34] [INFO]   |-- 👀 Checking 'nvidia/gliner-pii' in provider named 'nvidia' for model alias 'gliner-pii-detector'...
[12:08:34] [INFO]   |-- ✅ Passed!
[12:08:34] [INFO] 🌱 Sampling 10 records from seed dataset
[12:08:34] [INFO]   |-- seed dataset size: 

Entity,Label,Sensitivity,Protection
Mara,first_name,high,replace
Delgado,last_name,high,replace
34,age,medium,generalize
bartender,occupation,low,left_as_is
The Lantern,company_name,medium,generalize
Asheville,city,medium,generalize
North Carolina,state,low,left_as_is
Raleigh,city,medium,generalize
degree in hospitality management,education_level,low,left_as_is
Luis,first_name,medium,replace


In [5]:
preview.display_record(1)

IndexError: Record index 1 is out of bounds for 1 records.

In [ ]:
# preview.trace_dataframe.to_csv("temp_results_rewrite.csv")

In [ ]:
preview

PreviewResult(rows=2, columns=6, trace_columns=43, failed_records=0, preview_num_records=2)

## Inspect the results

`result.dataframe` has user-facing columns only.
`result.trace_dataframe` has every intermediate column for debugging.

In [ ]:
config = AnonymizerConfig(rewrite=Rewrite())
result = anonymizer.run(config=config, data=input_data)

print(result)
result.dataframe.head()

[02:14:56] [INFO] 📂 Loaded 10 records from /Users/lramaswamy/Documents/github/Anonymizer/docs/notebook_source/data/synth_bios_sample10.csv (column: 'bio')
[02:14:56] [DEBUG] input text lengths: min=1529, max=1843, mean=1661 chars (10 records)
[02:14:56] [DEBUG] detection config: threshold=0.30, labels=(default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)
[02:14:56] [INFO] 🔍 Running entity detection on 10 records
[02:14:56] [DEBUG] detection aliases: detector=gliner-pii-detector, validator=gpt-oss-120b, augmenter=gpt-oss-120b
[02:14:56] [DEBUG] NDD workflow 'entity-detection' starting with 10 records
[02:14:56] [DEBUG] NDD workflow 'entity-detection': 10 columns ['_raw_detected_entities', '_seed_entities', '_validation_candidates', '_validation_skeleton', '_validation_decisions', '_validated_entities', '_seed_entities_json', '_augmented_entities', '_merged_entities', '_detected_entities']
[02:14:56] [INFO] 🎨 Creating Data Designer dataset
[02:14:56] [INFO] ✅ Validation pas

AnonymizerResult(rows=10, columns=6, trace_columns=43, failed_records=0)


,bio,bio_rewritten,utility_score,leakage_mass,any_high_leaked,needs_human_review
0,"Mara Delgado, a 34‑year‑old bartender, has bec...","Elena Khan, a bartender in her mid‑30s, has be...",0.89,0.0,False,False
1,"Dr. Maya Patel, 38, is a pediatric cardiologis...","Dr. Anika Shah, in her late 30s, is a pediatri...",0.961538,0.6,False,False
2,"Mira Alvarez, 38, is an independent film direc...","Elena Khan, in their late 30s, is a filmmaker ...",0.95,0.6,False,False
3,"James “Jim” Alvarez, 34, has been soaring the ...","Ethan “Rex” Kumar, in their mid‑30s, has been ...",0.914286,0.9,False,False
4,"Lena Torres, 31, is a senior game designer at ...","Sofia Khan, in her early 30s, is a senior game...",0.84,0.0,False,False


In [ ]:
result.dataframe

,bio,bio_rewritten,utility_score,leakage_mass,any_high_leaked,needs_human_review
0,"Mara Delgado, a 34‑year‑old bartender, has bec...","Elena Khan, a bartender in her mid‑30s, has be...",0.89,0.0,False,False
1,"Dr. Maya Patel, 38, is a pediatric cardiologis...","Dr. Anika Shah, in her late 30s, is a pediatri...",0.961538,0.6,False,False
2,"Mira Alvarez, 38, is an independent film direc...","Elena Khan, in their late 30s, is a filmmaker ...",0.95,0.6,False,False
3,"James “Jim” Alvarez, 34, has been soaring the ...","Ethan “Rex” Kumar, in their mid‑30s, has been ...",0.914286,0.9,False,False
4,"Lena Torres, 31, is a senior game designer at ...","Sofia Khan, in her early 30s, is a senior game...",0.84,0.0,False,False
5,"Maya Patel, 42, is a tenured professor of Earl...","Sofia Khan, in her early 40s, is a tenured pro...",0.896429,0.6,False,False
6,Maya Patel is a 34‑year‑old lead gameplay prog...,Priya Shah is a lead gameplay programmer in he...,0.954545,0.6,False,False
7,Maya Torres is a 34‑year‑old bartender who has...,Nina Garcia is a bartender in her mid‑30s who ...,1.0,0.0,False,False
8,Megan “Megs” Alvarez is a 38‑year‑old executiv...,Sofia “Sofi” Khan is in her late 30s and works...,0.93,0.0,False,False
9,"Elena Morales, 42, is a senior economist at th...","Isabel Vargas, in her early 40s, is a senior e...",0.954545,0.6,False,False


In [ ]:
result.trace_dataframe.final_entities[0]

{'entities': [{'id': 'first_name_0_4',
   'value': 'Mara',
   'label': 'first_name',
   'start_position': 0,
   'end_position': 4,
   'score': 1.0,
   'source': 'detector'},
  {'id': 'last_name_5_12',
   'value': 'Delgado',
   'label': 'last_name',
   'start_position': 5,
   'end_position': 12,
   'score': 1.0,
   'source': 'detector'},
  {'id': 'age_16_18',
   'value': '34',
   'label': 'age',
   'start_position': 16,
   'end_position': 18,
   'score': 0.996,
   'source': 'detector'},
  {'id': 'occupation_28_37',
   'value': 'bartender',
   'label': 'occupation',
   'start_position': 28,
   'end_position': 37,
   'score': 0.999,
   'source': 'detector'},
  {'id': 'company_name_102_113',
   'value': 'The Lantern',
   'label': 'company_name',
   'start_position': 102,
   'end_position': 113,
   'score': 1.0,
   'source': 'augmenter'},
  {'id': 'city_184_193',
   'value': 'Asheville',
   'label': 'city',
   'start_position': 184,
   'end_position': 193,
   'score': 1.0,
   'source': 'det

## Custom privacy goal

For domain-specific data you can spell out exactly what to protect
and what to preserve.

In [ ]:
from anonymizer.config.rewrite import EvaluationCriteria, PrivacyGoal, RiskTolerance

custom_config = AnonymizerConfig(
    rewrite=Rewrite(
        privacy_goal=PrivacyGoal(
            protect="All direct identifiers, quasi-identifier combinations, and medical record numbers",
            preserve="Clinical meaning, temporal relationships, and treatment outcomes",
        ),
        evaluation=EvaluationCriteria(
            risk_tolerance=RiskTolerance.strict,
            max_repair_iterations=3,
        ),
    ),
)

custom_preview = anonymizer.preview(
    config=custom_config,
    data=input_data,
    num_records=3,
)

custom_preview.display_record(0)

[02:22:40] [INFO] 📂 Loaded 10 records from /Users/lramaswamy/Documents/github/Anonymizer/docs/notebook_source/data/synth_bios_sample10.csv (column: 'bio')
[02:22:40] [DEBUG] input text lengths: min=1529, max=1843, mean=1661 chars (10 records)
[02:22:40] [DEBUG] detection config: threshold=0.30, labels=(default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)
[02:22:40] [INFO]   |-- 👀 Preview mode: processing 3 of 10 records
[02:22:40] [INFO] 🔍 Running entity detection on 3 records
[02:22:40] [DEBUG] detection aliases: detector=gliner-pii-detector, validator=gpt-oss-120b, augmenter=gpt-oss-120b
[02:22:40] [DEBUG] NDD workflow 'entity-detection' starting with 10 records
[02:22:40] [DEBUG] NDD workflow 'entity-detection': 10 columns ['_raw_detected_entities', '_seed_entities', '_validation_candidates', '_validation_skeleton', '_validation_decisions', '_validated_entities', '_seed_entities_json', '_augmented_entities', '_merged_entities', '_detected_entities']
[02:22:40] [INFO] 👀

Entity,Label,Sensitivity,Protection
34,age,medium,generalize
Asheville,city,medium,generalize
Delgado,last_name,high,replace
Luis,first_name,medium,replace
Mara,first_name,high,replace
North Carolina,state,low,left_as_is
Raleigh,city,medium,generalize
Rosa,first_name,medium,replace
bartender,occupation,low,left_as_is
degree in hospitality management,education_level,medium,left_as_is


## Filter by review flag

Records where automated metrics exceed thresholds are flagged for manual review.

In [ ]:
df = result.dataframe
flagged = df[df["needs_human_review"] == True]  # noqa: E712
print(f"{len(flagged)} of {len(df)} records flagged for human review")
flagged.head()

0 of 10 records flagged for human review


,bio,bio_rewritten,utility_score,leakage_mass,any_high_leaked,needs_human_review
